# Notebook 2 — Radiomics Feature Extraction & Analysis
## ML4SCI / CERN PrediCT Evaluation Test — Project 2

**Author:** Aditya Parashar  
**Dataset:** COCA — Coronary Calcium and Chest CTs (Stanford AIMI)

**References:**
1. **Agatston scoring:** Agatston AS et al. *JACC* 1990;15(4):827-832.
2. **PyRadiomics:** van Griethuysen JJM et al. *Cancer Research* 2017;77(21):e104-e107.
3. **IBSI:** Zwanenburg A et al. *Radiology* 2020.
4. **COCA dataset:** Eng D et al. *npj Digital Medicine* 2021;4(1):88. DOI: 10.1038/s41746-021-00460-1
5. **Cardiac radiomics:** Ayx I et al. *Diagnostics* 2023;13(2):307.

---
## Section 1 — Setup & Kernel Verification

> **IMPORTANT:** This notebook requires the `predict_env` kernel (Python 3.8) with PyRadiomics installed.
> If you see an `ImportError`, go to Kernel → Change Kernel → select "PrediCT Python 3.8".

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import SimpleITK as sitk
import cv2
import plistlib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)


print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

import radiomics
from radiomics import featureextractor
import logging
radiomics.setVerbosity(logging.ERROR)
print(f"PyRadiomics version: {radiomics.__version__}")


DATA_ROOT = r"data"
print(f"DATA_ROOT: {os.path.abspath(DATA_ROOT)}")

Python executable: c:\Users\Aditya\miniconda3\envs\predict_env\python.exe
Python version: 3.8.20 (default, Oct  3 2024, 15:19:54) [MSC v.1929 64 bit (AMD64)]
PyRadiomics version: 3.1.0a2
DATA_ROOT: d:\ml4sci2\PrediCT\data


In [ ]:
# Display params.yaml
print("=" * 60)
print("PyRadiomics Configuration (params.yaml)")
print("=" * 60)
with open('params.yaml', 'r') as f:
    print(f.read())

PyRadiomics Configuration (params.yaml)
imageType:
  Original: {}

featureClass:
  shape:
    - Sphericity
    - SurfaceVolumeRatio
    - Maximum3DDiameter
  glcm:
    - Contrast
    - Correlation
    - Idm
  glszm:
    - SmallAreaEmphasis
    - LargeAreaEmphasis
    - ZonePercentage
  glrlm:
    - ShortRunEmphasis
    - LongRunEmphasis
    - RunPercentage

setting:
  minimumROIDimensions: 1
  minimumROISize: 1
  correctMask: true
  geometryTolerance: 1.0e-4
  binWidth: 25
  resampledPixelSpacing: [1.0, 1.0, 1.0]
  interpolator: sitkBSpline
  label: 1



## Methodology & Compliance Statement

Feature extraction was performed using **PyRadiomics v3.x** in accordance with 
**IBSI Chapter 1** (Image Biomarker Standardisation Initiative) guidelines.

| Parameter | Value | Justification |
|-----------|-------|---------------|
| Resampling | 1.0 × 1.0 × 1.0 mm³ isotropic | Required for accurate shape features |
| Interpolation (CT) | B-spline | Best HU value preservation for CT |
| Interpolation (mask) | Nearest-neighbor | Preserves binary mask integrity |
| Intensity discretization | Fixed bin width = 25 HU | IBSI recommendation for CT |
| Feature computation | 3D volumetric | Full 3D spatial context |
| ROI definition | Expert-annotated XML contours (COCA dataset) | Ground truth, not thresholding |
| Statistical correction | Benjamini-Hochberg FDR | Controls false discovery rate across 15 features |

Agatston scores computed per original methodology (Agatston et al., JACC 1990) 
using ground truth lesion areas and peak HU values from radiologist annotations.

---
## Data Loading Functions (from Notebook 1)

Re-implemented here for self-contained execution.

In [3]:
def discover_patients(data_root):
    raw_dir = os.path.join(data_root, 'raw')
    xml_dir = os.path.join(data_root, 'calcium_xml')
    raw_ids = {n for n in os.listdir(raw_dir) if os.path.isdir(os.path.join(raw_dir, n))}
    xml_ids = {n.replace('.xml', '') for n in os.listdir(xml_dir) if n.endswith('.xml')}
    valid_ids = sorted(raw_ids & xml_ids, key=lambda x: int(x))
    xml_only = xml_ids - raw_ids
    if xml_only:
        print(f"WARNING: XML exists but no DICOM folder for patient(s): "
              f"{sorted(xml_only, key=lambda x: int(x))} — excluded from processing")
    print(f"Discovered {len(valid_ids)} valid patients")
    return valid_ids

def find_dicom_subfolder(patient_dir):
    for name in os.listdir(patient_dir):
        sub = os.path.join(patient_dir, name)
        if os.path.isdir(sub) and any(f.endswith('.dcm') for f in os.listdir(sub)):
            return sub
    if any(f.endswith('.dcm') for f in os.listdir(patient_dir)):
        return patient_dir
    raise FileNotFoundError(f"No DICOM files found in {patient_dir}")

def load_patient_dicom(patient_id, data_root):
    import pydicom
    patient_dir = os.path.join(data_root, 'raw', str(patient_id))
    dcm_dir = find_dicom_subfolder(patient_dir)
    dcm_files = [os.path.join(dcm_dir, f) for f in os.listdir(dcm_dir) if f.endswith('.dcm')]
    slices = []
    for f in dcm_files:
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        slices.append((int(ds.InstanceNumber), f))
    slices.sort(key=lambda x: x[0])
    sorted_files = [f for _, f in slices]
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(sorted_files)
    ct_sitk = reader.Execute()
    return ct_sitk

def parse_calcium_xml(patient_id, data_root):
    xml_path = os.path.join(data_root, 'calcium_xml', f'{patient_id}.xml')
    with open(xml_path, 'rb') as f:
        plist_data = plistlib.load(f)
    annotations = []
    for image_entry in plist_data.get('Images', []):
        slice_info = {'ImageIndex': image_entry['ImageIndex'], 'ROIs': []}
        for roi in image_entry.get('ROIs', []):
            if roi.get('NumberOfPoints', 0) == 0:
                continue
            slice_info['ROIs'].append({
                'Area': roi.get('Area', 0.0), 'Max': roi.get('Max', 0.0),
                'Mean': roi.get('Mean', 0.0), 'Min': roi.get('Min', 0.0),
                'Dev': roi.get('Dev', 0.0), 'Total': roi.get('Total', 0.0),
                'Name': roi.get('Name', 'Unknown'),
                'NumberOfPoints': roi.get('NumberOfPoints', 0),
                'Point_px': roi.get('Point_px', [])
            })
        annotations.append(slice_info)
    return annotations

def build_3d_mask(annotations, ct_sitk):
    ct_array = sitk.GetArrayFromImage(ct_sitk)
    n_slices, height, width = ct_array.shape
    mask_array = np.zeros((n_slices, height, width), dtype=np.uint8)
    for entry in annotations:
        slice_idx = entry['ImageIndex']
        if slice_idx >= n_slices:
            continue
        for roi in entry['ROIs']:
            if roi['NumberOfPoints'] == 0:
                continue
            points = []
            for pt_str in roi['Point_px']:
                x_str, y_str = pt_str.strip('()').split(',')
                points.append([int(round(float(x_str.strip()))), int(round(float(y_str.strip())))])
            if len(points) >= 3:
                pts = np.array(points, dtype=np.int32).reshape((-1, 1, 2))
                cv2.fillPoly(mask_array[slice_idx], [pts], color=1)
    mask_sitk = sitk.GetImageFromArray(mask_array)
    mask_sitk.CopyInformation(ct_sitk)
    return mask_sitk, mask_array

def resample_to_isotropic(ct_sitk, mask_sitk=None, target_spacing=(1.0, 1.0, 1.0)):
    original_spacing = ct_sitk.GetSpacing()
    original_size = ct_sitk.GetSize()
    new_size = [int(round(osz * ospc / tspc))
                for osz, ospc, tspc in zip(original_size, original_spacing, target_spacing)]
    resample = sitk.ResampleImageFilter()
    resample.SetOutputSpacing(target_spacing)
    resample.SetSize(new_size)
    resample.SetOutputDirection(ct_sitk.GetDirection())
    resample.SetOutputOrigin(ct_sitk.GetOrigin())
    resample.SetTransform(sitk.Transform())
    resample.SetInterpolator(sitk.sitkBSpline)
    resample.SetDefaultPixelValue(-1000)
    ct_resampled = resample.Execute(ct_sitk)
    if mask_sitk is None:
        return ct_resampled
    resample.SetInterpolator(sitk.sitkNearestNeighbor)
    resample.SetDefaultPixelValue(0)
    mask_resampled = resample.Execute(mask_sitk)
    return ct_resampled, mask_resampled

patient_ids = discover_patients(DATA_ROOT)

Discovered 30 valid patients


---
## Section 2 — PyRadiomics Feature Extraction

### Six Known Failure Modes (All Handled)
| # | Error | Fix |
|---|-------|-----|
| 1 | Geometry mismatch | `mask_sitk.CopyInformation(ct_sitk)` — always |
| 2 | Empty mask crash | Check `mask.sum() == 0` before extraction |
| 3 | Wrong mask dtype | Force `np.uint8`, label value = 1 |
| 4 | YAML tabs | params.yaml uses spaces only |
| 5 | Tiny ROI | `minimumROIDimensions: 1` in params.yaml |
| 6 | Wrong kernel | Verified above with `radiomics.__version__` |

In [ ]:
# Initialize extractor ONCE — reuse for all patients
extractor = featureextractor.RadiomicsFeatureExtractor('params.yaml')
print("Extractor initialized with params.yaml")
print(f"Enabled feature classes: {list(extractor.enabledFeatures.keys())}")

def extract_patient_features(ct_sitk, mask_sitk, patient_id):
    """Extract PyRadiomics features for a single patient.
    
    Handles all 6 known failure modes.
    
    Returns:
        dict of feature_name -> value, or None on failure/empty mask
    """
    
    mask_sitk.CopyInformation(ct_sitk)
    
    
    mask_array = sitk.GetArrayFromImage(mask_sitk)
    if mask_array.sum() == 0:
        print(f"  Patient {patient_id}: empty calcium mask — Agatston=0, features=NaN")
        return None
    
    
    if mask_array.dtype != np.uint8:
        mask_array = mask_array.astype(np.uint8)
        mask_sitk = sitk.GetImageFromArray(mask_array)
        mask_sitk.CopyInformation(ct_sitk)
    
    try:
        result = extractor.execute(ct_sitk, mask_sitk, label=1)
        
        features = {}
        for k, v in result.items():
            if not k.startswith('diagnostics_'):
                try:
                    features[k] = float(v)
                except (TypeError, ValueError):
                    features[k] = v
        return features
    except Exception as e:
        print(f"  Patient {patient_id}: extraction FAILED — {type(e).__name__}: {e}")
        return None

Extractor initialized with params.yaml
Enabled feature classes: ['shape', 'glcm', 'glszm', 'glrlm']


In [ ]:
# Extract features for ALL patients
print("=" * 70)
print("PyRadiomics Feature Extraction")
print("=" * 70)

all_features = []
feature_names = None
n_empty = 0
n_failed = 0

for pid in tqdm(patient_ids, desc="Extracting features"):
    try:
        
        ct_sitk = load_patient_dicom(pid, DATA_ROOT)
        annotations = parse_calcium_xml(pid, DATA_ROOT)
        mask_sitk, mask_array = build_3d_mask(annotations, ct_sitk)
        ct_resampled, mask_resampled = resample_to_isotropic(ct_sitk, mask_sitk)
        
        
        features = extract_patient_features(ct_resampled, mask_resampled, pid)
        
        if features is not None:
            if feature_names is None:
                feature_names = list(features.keys())
            row = {'patient_id': pid, **features}
        else:
            n_empty += 1
            if feature_names is not None:
                row = {'patient_id': pid, **{k: np.nan for k in feature_names}}
            else:
                row = {'patient_id': pid}
        
        all_features.append(row)
        
    except Exception as e:
        print(f"  Patient {pid}: LOAD FAILED — {e}")
        n_failed += 1
        if feature_names is not None:
            all_features.append({'patient_id': pid, **{k: np.nan for k in feature_names}})
        else:
            all_features.append({'patient_id': pid})

features_df = pd.DataFrame(all_features)
print(f"\nExtraction complete:")
print(f"  Total patients: {len(patient_ids)}")
print(f"  Successful: {len(patient_ids) - n_empty - n_failed}")
print(f"  Empty masks (NaN rows): {n_empty}")
print(f"  Failed: {n_failed}")
if feature_names:
    print(f"  Features extracted: {len(feature_names)}")
    print(f"  Feature names: {feature_names}")

PyRadiomics Feature Extraction


Extracting features: 100%|██████████| 30/30 [01:52<00:00,  3.76s/it]


Extraction complete:
  Total patients: 30
  Successful: 30
  Empty masks (NaN rows): 0
  Failed: 0
  Features extracted: 12
  Feature names: ['original_shape_Sphericity', 'original_shape_SurfaceVolumeRatio', 'original_shape_Maximum3DDiameter', 'original_glcm_Contrast', 'original_glcm_Correlation', 'original_glcm_Idm', 'original_glszm_SmallAreaEmphasis', 'original_glszm_LargeAreaEmphasis', 'original_glszm_ZonePercentage', 'original_glrlm_ShortRunEmphasis', 'original_glrlm_LongRunEmphasis', 'original_glrlm_RunPercentage']


---
## Section 3 — Optional Features from XML

Additional clinically relevant features derived directly from the XML annotation data.

In [ ]:
def compute_xml_features(patient_id, data_root, ct_sitk):
    """Compute optional features from XML annotations."""
    annotations = parse_calcium_xml(patient_id, data_root)
    
    all_max_hu = []
    all_mean_hu = []
    all_areas = []
    
    for entry in annotations:
        for roi in entry['ROIs']:
            if roi['NumberOfPoints'] == 0:
                continue
            all_max_hu.append(roi['Max'])
            all_mean_hu.append(roi['Mean'])
            all_areas.append(roi['Area'])
    
    if not all_max_hu:
        return {'max_calcium_hu': np.nan, 'mean_calcium_hu': np.nan, 
                'total_calcium_volume_mm3': 0.0}
    
    
    _, mask_arr = build_3d_mask(annotations, ct_sitk)
    spacing = ct_sitk.GetSpacing()
    voxel_volume_mm3 = spacing[0] * spacing[1] * spacing[2]
    total_vol = float(mask_arr.sum()) * voxel_volume_mm3
    
    return {
        'max_calcium_hu': max(all_max_hu),
        'mean_calcium_hu': np.average(all_mean_hu, weights=all_areas) if all_areas else np.nan,
        'total_calcium_volume_mm3': round(total_vol, 2)
    }


xml_feature_records = []
for pid in tqdm(patient_ids, desc="Computing XML features"):
    try:
        ct_sitk = load_patient_dicom(pid, DATA_ROOT)
        xml_feats = compute_xml_features(pid, DATA_ROOT, ct_sitk)
        xml_feats['patient_id'] = pid
        xml_feature_records.append(xml_feats)
    except Exception as e:
        xml_feature_records.append({
            'patient_id': pid, 'max_calcium_hu': np.nan,
            'mean_calcium_hu': np.nan, 'total_calcium_volume_mm3': np.nan
        })

xml_features_df = pd.DataFrame(xml_feature_records)
print("\nXML-derived features:")
print(xml_features_df.to_string(index=False))

Computing XML features: 100%|██████████| 30/30 [00:13<00:00,  2.25it/s]


XML-derived features:
 max_calcium_hu  mean_calcium_hu  total_calcium_volume_mm3 patient_id
          206.0       160.769226                     20.95          0
          934.0       307.427235                    490.19          1
          536.0       222.181591                    201.92          2
          719.0       289.499420                    539.76          3
          319.0       171.470588                     91.48          4
         1390.0       369.750701                   2387.20          5
          616.0       252.804732                    476.56          6
          367.0       206.844221                     85.99          7
          801.0       340.721888                     88.84          8
          619.0       245.565215                    108.42          9
          386.0       198.616602                    281.71         10
          540.0       246.453512                    220.22         11
          936.0       294.480642                   1991.82         

---
## Section 4 — Compile Feature Matrix

Merge PyRadiomics features with Agatston scores and XML features.  
Patients with failed extraction are kept with NaN values (not dropped).

In [ ]:

agatston_df = pd.read_csv('outputs/agatston_scores.csv')
print(f"Loaded Agatston scores: {len(agatston_df)} patients")


features_df['patient_id'] = features_df['patient_id'].astype(str)
agatston_df['patient_id'] = agatston_df['patient_id'].astype(str)
xml_features_df['patient_id'] = xml_features_df['patient_id'].astype(str)


merged_df = features_df.merge(agatston_df, on='patient_id', how='left')
merged_df = merged_df.merge(xml_features_df, on='patient_id', how='left')


merged_df.to_csv('outputs/radiomics_features.csv', index=False)
print(f"\nSaved outputs/radiomics_features.csv")
print(f"  Shape: {merged_df.shape} ({merged_df.shape[0]} patients × {merged_df.shape[1]} columns)")
print(f"  Patients with NaN features: {merged_df[feature_names].isna().all(axis=1).sum() if feature_names else 'N/A'}")
print(f"\nColumn names: {list(merged_df.columns)}")

Loaded Agatston scores: 30 patients

Saved outputs/radiomics_features.csv
  Shape: (30, 21) (30 patients × 21 columns)
  Patients with NaN features: 0

Column names: ['patient_id', 'original_shape_Sphericity', 'original_shape_SurfaceVolumeRatio', 'original_shape_Maximum3DDiameter', 'original_glcm_Contrast', 'original_glcm_Correlation', 'original_glcm_Idm', 'original_glszm_SmallAreaEmphasis', 'original_glszm_LargeAreaEmphasis', 'original_glszm_ZonePercentage', 'original_glrlm_ShortRunEmphasis', 'original_glrlm_LongRunEmphasis', 'original_glrlm_RunPercentage', 'agatston_score', 'category', 'score_rca', 'score_lad', 'score_lcx', 'max_calcium_hu', 'mean_calcium_hu', 'total_calcium_volume_mm3']


---
## Section 5 — Statistical Analysis

**Methods:**
- **Spearman correlation** (ρ) between each feature and continuous Agatston score (non-parametric)
- **Kruskal-Wallis test** (H) across 4 Agatston categories for each feature
- **FDR correction** (Benjamini-Hochberg) for multiple comparisons — required when testing >10 features

*Per IBSI guidelines, both uncorrected p-values and FDR-corrected q-values are reported.*

In [ ]:
from statsmodels.stats.multitest import multipletests

def run_statistical_analysis(df, feature_cols, agatston_col='agatston_score', 
                              category_col='category'):
    """Complete statistical analysis: Spearman + Kruskal-Wallis + FDR."""
    
    df_clean = df.dropna(subset=feature_cols)
    n_excluded = len(df) - len(df_clean)
    print(f"Analyzing {len(df_clean)} patients (excluded {n_excluded} with incomplete features)")
    
    results = []
    for feat in feature_cols:
        
        rho, p_sp = stats.spearmanr(df_clean[feat], df_clean[agatston_col])
        
        
        groups = [df_clean[df_clean[category_col] == cat][feat].values 
                  for cat in df_clean[category_col].unique()]
        groups = [g for g in groups if len(g) > 0]
        
        if len(groups) >= 2:
            h_stat, p_kw = stats.kruskal(*groups)
        else:
            h_stat, p_kw = np.nan, np.nan
        
        results.append({
            'feature': feat,
            'spearman_rho': round(rho, 4),
            'spearman_p': p_sp,
            'kw_H': round(h_stat, 4) if not np.isnan(h_stat) else np.nan,
            'kw_p': p_kw
        })
    
    res_df = pd.DataFrame(results)
    
    
    valid_mask = res_df['spearman_p'].notna()
    if valid_mask.sum() > 0:
        _, q_vals, _, _ = multipletests(
            res_df.loc[valid_mask, 'spearman_p'].values, method='fdr_bh')
        res_df.loc[valid_mask, 'fdr_q'] = q_vals
    else:
        res_df['fdr_q'] = np.nan
    
    
    res_df['sig'] = res_df['spearman_p'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else '')))
    
    return res_df.sort_values('spearman_p')


metadata_cols = ['patient_id', 'agatston_score', 'category', 'score_rca', 'score_lad', 
                 'score_lcx', 'max_calcium_hu', 'mean_calcium_hu', 'total_calcium_volume_mm3']
all_feature_cols = [c for c in merged_df.columns if c not in metadata_cols]


analysis_cols = all_feature_cols + ['max_calcium_hu', 'mean_calcium_hu', 'total_calcium_volume_mm3']
analysis_cols = [c for c in analysis_cols if c in merged_df.columns]

results_df = run_statistical_analysis(merged_df, analysis_cols)


print("\n" + "=" * 90)
print("STATISTICAL ANALYSIS RESULTS")
print("=" * 90)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 40)
print(results_df.to_string(index=False))


n_sig_p = (results_df['spearman_p'] < 0.05).sum()
n_sig_q = (results_df['fdr_q'] < 0.05).sum() if 'fdr_q' in results_df.columns else 0
print(f"\nSignificant features (p < 0.05 uncorrected): {n_sig_p}/{len(results_df)}")
print(f"Significant features (q < 0.05 FDR-corrected): {n_sig_q}/{len(results_df)}")

Analyzing 30 patients (excluded 0 with incomplete features)

STATISTICAL ANALYSIS RESULTS
                          feature  spearman_rho   spearman_p    kw_H     kw_p        fdr_q sig
         total_calcium_volume_mm3        0.9862 2.031047e-23 24.7259 0.000004 3.046571e-22 ***
                   max_calcium_hu        0.9012 1.116793e-11 18.0577 0.000120 8.375949e-11 ***
 original_shape_Maximum3DDiameter        0.8806 1.387646e-10 20.6203 0.000033 6.938232e-10 ***
        original_shape_Sphericity       -0.8287 1.557208e-08 19.8472 0.000049 5.839529e-08 ***
original_shape_SurfaceVolumeRatio       -0.8167 3.716568e-08 15.2782 0.000481 1.114971e-07 ***
                  mean_calcium_hu        0.7139 9.436241e-06  9.5410 0.008476 2.359060e-05 ***
           original_glcm_Contrast        0.6885 2.596769e-05 12.3207 0.002111 5.564505e-05 ***
        original_glcm_Correlation        0.6761 4.119370e-05 17.2773 0.000177 7.723818e-05 ***
 original_glszm_LargeAreaEmphasis        0.6583 7.67816

---
## Section 6 — Visualizations

All plots saved to `outputs/` directory.

In [ ]:
# Correlation Matrix Heatmap Plot
df_for_corr = merged_df[analysis_cols].dropna()
if len(df_for_corr) > 2 and len(df_for_corr.columns) > 1:
    corr_matrix = df_for_corr.corr(method='spearman')
    
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                square=True, linewidths=0.5, ax=ax,
                xticklabels=[c.split('_')[-1] if 'original' in c else c for c in corr_matrix.columns],
                yticklabels=[c.split('_')[-1] if 'original' in c else c for c in corr_matrix.columns])
    ax.set_title('Feature Correlation Matrix (Spearman)', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/correlation_heatmap.png")
else:
    print("Not enough data for correlation heatmap")

Saved: outputs/correlation_heatmap.png


In [ ]:
# Top 10 Features by |Spearman ρ| Plot
top_n = min(10, len(results_df))
top_features = results_df.nlargest(top_n, 'spearman_rho', keep='first').copy()
top_features['abs_rho'] = top_features['spearman_rho'].abs()
top_features = top_features.sort_values('abs_rho', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.5)))
colors = ['#F44336' if r < 0 else '#2196F3' for r in top_features['spearman_rho']]
ax.barh(range(len(top_features)), top_features['spearman_rho'], color=colors, edgecolor='white')
ax.set_yticks(range(len(top_features)))


short_names = []
for name in top_features['feature']:
    if 'original_' in name:
        short = name.split('original_')[-1]
    else:
        short = name
    short_names.append(short[:35])
ax.set_yticklabels(short_names, fontsize=10)

ax.set_xlabel('Spearman ρ', fontsize=13)
ax.set_title(f'Top {top_n} Features by Spearman Correlation with Agatston Score', 
             fontsize=14, fontweight='bold')
ax.axvline(x=0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.savefig('outputs/top_features_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/top_features_bar.png")

Saved: outputs/top_features_bar.png


In [ ]:
# Scatter plots for top 3 correlated features Plot
top3 = results_df.head(3)
df_plot = merged_df.dropna(subset=analysis_cols)

fig, axes = plt.subplots(1, min(3, len(top3)), figsize=(6 * min(3, len(top3)), 5))
if len(top3) == 1:
    axes = [axes]

for i, (_, row) in enumerate(top3.iterrows()):
    if i >= 3:
        break
    feat = row['feature']
    ax = axes[i]
    
    
    cat_colors = {'0': '#4CAF50', '1-99': '#FFC107', '100-399': '#FF9800', '400+': '#F44336'}
    for cat, color in cat_colors.items():
        mask = df_plot['category'] == cat
        if mask.any():
            ax.scatter(df_plot.loc[mask, feat], df_plot.loc[mask, 'agatston_score'],
                      c=color, label=cat, alpha=0.7, s=60, edgecolors='white')
    
    
    x_vals = df_plot[feat].values
    y_vals = df_plot['agatston_score'].values
    valid = ~(np.isnan(x_vals) | np.isnan(y_vals))
    if valid.sum() > 2:
        z = np.polyfit(x_vals[valid], y_vals[valid], 1)
        p = np.poly1d(z)
        x_line = np.linspace(x_vals[valid].min(), x_vals[valid].max(), 100)
        ax.plot(x_line, p(x_line), 'k--', alpha=0.5, linewidth=1.5)
    
    short_name = feat.split('original_')[-1] if 'original_' in feat else feat
    ax.set_xlabel(short_name[:30], fontsize=11)
    ax.set_ylabel('Agatston Score', fontsize=11)
    ax.set_title(f'ρ={row["spearman_rho"]:.3f}, p={row["spearman_p"]:.4f}', fontsize=11)
    ax.legend(title='Category', fontsize=9)

plt.suptitle('Top 3 Features vs Agatston Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/top3_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/top3_scatter.png")

Saved: outputs/top3_scatter.png


In [ ]:
# t-SNE Visualization Plot
RANDOM_STATE = 42


df_tsne = merged_df.dropna(subset=all_feature_cols)
n_excluded_tsne = len(merged_df) - len(df_tsne)
print(f"t-SNE: Using {len(df_tsne)} patients (excluded {n_excluded_tsne} with NaN features)")

if len(df_tsne) >= 5:
    
    X_tsne = StandardScaler().fit_transform(df_tsne[all_feature_cols].values)
    
    
    perplexity = min(5, len(df_tsne) - 1)
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=RANDOM_STATE, 
                n_iter=1000)
    embedding = tsne.fit_transform(X_tsne)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    cat_colors = {'0': '#4CAF50', '1-99': '#FFC107', '100-399': '#FF9800', '400+': '#F44336'}
    
    for cat, color in cat_colors.items():
        mask = df_tsne['category'].values == cat
        if mask.any():
            ax.scatter(embedding[mask, 0], embedding[mask, 1], c=color, label=cat,
                      s=100, alpha=0.8, edgecolors='white', linewidths=1.5)
    
    ax.set_xlabel('t-SNE 1', fontsize=13)
    ax.set_ylabel('t-SNE 2', fontsize=13)
    ax.set_title(f't-SNE of Radiomics Features (perplexity={perplexity}, '
                 f'random_state={RANDOM_STATE})', fontsize=14, fontweight='bold')
    ax.legend(title='Agatston Category', fontsize=11, title_fontsize=12)
    plt.tight_layout()
    plt.savefig('outputs/tsne.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/tsne.png")
else:
    print(f"Too few patients ({len(df_tsne)}) for t-SNE visualization")

t-SNE: Using 30 patients (excluded 0 with NaN features)
Saved: outputs/tsne.png


In [ ]:
# KMeans Clustering + Adjusted Rand Index Plot
if len(df_tsne) >= 5:
    X_cluster = StandardScaler().fit_transform(df_tsne[all_feature_cols].values)
    
    kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
    cluster_labels = kmeans.fit_predict(X_cluster)
    
    
    cat_map = {'0': 0, '1-99': 1, '100-399': 2, '400+': 3}
    true_labels = df_tsne['category'].map(cat_map).values
    
    ari = adjusted_rand_score(true_labels, cluster_labels)
    print(f"KMeans (k=4) vs Agatston Categories:")
    print(f"  Adjusted Rand Index (ARI) = {ari:.4f}")
    print(f"  ARI interpretation: {'Good agreement' if ari > 0.5 else 'Moderate agreement' if ari > 0.2 else 'Weak agreement'}")
    
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    
    for cat, color in cat_colors.items():
        mask = df_tsne['category'].values == cat
        if mask.any():
            axes[0].scatter(embedding[mask, 0], embedding[mask, 1], c=color, label=cat,
                           s=100, alpha=0.8, edgecolors='white')
    axes[0].set_title('True Agatston Categories', fontsize=13, fontweight='bold')
    axes[0].legend(title='Category')
    
    
    cluster_colors = ['#1E88E5', '#E53935', '#43A047', '#FB8C00']
    for i in range(4):
        mask = cluster_labels == i
        if mask.any():
            axes[1].scatter(embedding[mask, 0], embedding[mask, 1], c=cluster_colors[i],
                           label=f'Cluster {i}', s=100, alpha=0.8, edgecolors='white')
    axes[1].set_title(f'KMeans Clusters (ARI={ari:.3f})', fontsize=13, fontweight='bold')
    axes[1].legend(title='Cluster')
    
    for ax in axes:
        ax.set_xlabel('t-SNE 1')
        ax.set_ylabel('t-SNE 2')
    
    plt.suptitle(f't-SNE: Categories vs Clusters (random_state={RANDOM_STATE})', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/tsne_clusters.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/tsne_clusters.png")
else:
    print("Too few patients for clustering analysis")

KMeans (k=4) vs Agatston Categories:
  Adjusted Rand Index (ARI) = 0.2000
  ARI interpretation: Moderate agreement
Saved: outputs/tsne_clusters.png


---
## Summary


This notebook implements a complete radiomics feature extraction and statistical analysis pipeline for coronary artery calcium assessment. Specifically, it:

1. Verifies the PyRadiomics environment configuration (Python 3.8 with the correct kernel).
2. Extracts radiomic features per patient, including shape, GLCM, GLSZM, and GLRLM features (12 features per patient).
3. Systematically handles six common failure modes: geometry inconsistencies, empty masks, data type issues, YAML configuration errors, extremely small ROIs, and kernel mismatches.
4. Optionally derives XML-based quantitative features, including maximum calcium HU, mean calcium HU, and total calcium volume (mm³).
5. Merges the resulting feature matrix with Agatston scores while preserving patients with missing values (NaNs).
6. Performs statistical analyses, including Spearman’s correlation (ρ), Kruskal–Wallis H testing, and false discovery rate (FDR) correction using the Benjamini–Hochberg procedure.
7. Generates visualization outputs such as a correlation heatmap, top-10 feature importance bar chart, and feature–score scatter plots.
8. Applies t-SNE dimensionality reduction (perplexity = 5, fixed random seed reported) for exploratory visualization.
9. Conducts KMeans clustering and evaluates agreement with Agatston risk categories using the Adjusted Rand Index (ARI).




**Outputs saved:**
- `outputs/radiomics_features.csv`
- `outputs/correlation_heatmap.png`
- `outputs/top_features_bar.png`
- `outputs/top3_scatter.png`
- `outputs/tsne.png`
- `outputs/tsne_clusters.png`

**IBSI Compliance:** Feature extraction performed per IBSI Chapter 1 guidelines using PyRadiomics.  
**Statistical Rigor:** FDR correction applied; both uncorrected and corrected p-values reported.

## References

1. **Agatston Scoring Methodology:**  
   Agatston AS, Janowitz WR, Hildner FJ, et al. Quantification of coronary artery 
   calcium using ultrafast computed tomography. *JACC* 1990;15(4):827–832.  
   DOI: 10.1016/0735-1097(90)90282-T

2. **PyRadiomics:**  
   van Griethuysen JJM, Fedorov A, Parmar C, et al. Computational Radiomics System 
   to Decode the Radiographic Phenotype. *Cancer Research* 2017;77(21):e104–e107.  
   DOI: 10.1158/0008-5472.CAN-17-0339

3. **IBSI — Image Biomarker Standardisation Initiative:**  
   Zwanenburg A, Vallières M, Abdalah MA, et al. The Image Biomarker Standardization 
   Initiative. *Radiology* 2020;295(2):328–338.  
   DOI: 10.1148/radiol.2020191145

4. **COCA Dataset:**  
   Eng D, Chute C, Khandwala N, et al. Automated coronary calcium scoring using deep 
   learning with multicenter external validation. *npj Digital Medicine* 2021;4(1):88.  
   DOI: 10.1038/s41746-021-00460-1

5. **Cardiac CT Radiomics:**  
   Ayx I, Tharmaseelan H, Hertel A, et al. Radiomics in Cardiac Computed Tomography. 
   *Diagnostics* 2023;13(2):307.  
   DOI: 10.3390/diagnostics13020307